# Specify a bespoke time dimension

This notebook demonstrates how to provide a bespoke `time_dim` to a temporal computation. By default the temporal methods detect the time dimension automatically from the metadata of the data object. However, when your data uses a non-standard name for the time coordinate, or contains more than one time-like coordinate, you can select the dimension to aggregate over explicitly using the `time_dim` argument.

In [ ]:
from earthkit import data as ekd
from earthkit import transforms as ekt
from earthkit.transforms._tools import earthkit_remote_test_data_file

remote_era5_file = earthkit_remote_test_data_file("era5-Europe-sfc-2m-temperature-3deg-2015-2017.grib")
era5_data = ekd.from_source("url", remote_era5_file)
ds = era5_data.to_xarray()

ds

## Rename the time coordinate

To simulate a dataset with a non-standard time coordinate name, we rename the `valid_time` coordinate to `forecast_time`. We also drop the CF `standard_name` and `axis` attributes so that the automatic detection can no longer identify it as the time dimension.

In [ ]:
time_coord = ekt._tools.get_dim_key(ds, "t")

ds_renamed = ds.rename({time_coord: "forecast_time"})
ds_renamed["forecast_time"].attrs.pop("standard_name", None)
ds_renamed["forecast_time"].attrs.pop("axis", None)

ds_renamed

## Provide the bespoke `time_dim`

Because the time coordinate is now named `forecast_time` and no longer carries the CF metadata used for automatic detection, we pass `time_dim="forecast_time"` to `temporal.daily_mean` so that it aggregates along the correct dimension.

In [ ]:
daily_mean = ekt.temporal.daily_mean(ds_renamed, time_dim="forecast_time")

daily_mean